# `neo4j_payments`: counterparty-id anatomy, checks, PCARD, cpty_type, and unmapped columns

One month, all 24 categories. Every table is an aggregate. Ids appear only as masked **shapes** (`D9-D20`, `H32`, `A3|D4-D2-D2|D14`), where
D = digits, A = letters, H = hex, X = mixed, and the number is the length. Delimiters are kept.

| § | Question |
|---|---|
| 0 | Full schema: which columns have we never mapped? |
| 1 | Population matrix: every column × every category |
| 2 | Enum columns (≤ 40 values): their values per category, `cpty_type` included |
| 3 | `unq_cpty_acct_id` vs `unq_cpty_id` (+ `merchant_id`): coverage, formats, is B a hash of something, entity stability, sharing across categories |
| 4 | Checks: is the 7B `trans_id` tail an account? Can payee names become ids? How many check payees are already paid electronically? 7C on-us test |
| 5 | PCARD / debit card: `merchant_id` vs `unq_cpty_id` vs name |
| 6 | `*_orig` columns vs their base column |

**Spark 3.3 only.** Runtime is several passes over ~420M rows. Set `PROBE_DAYS = 7` for a faster first look.

In [ ]:
# ---- Config -------------------------------------------------------------------------------
TABLE       = "dsihd01p_dsi.neo4j_payments"
PROBE_MONTH = "2025-01"
PROBE_DAYS  = None                 # e.g. 7 → first 7 days only

ID_A, ID_B, MERCH_COL = "unq_cpty_acct_id", "unq_cpty_id", "merchant_id"
CHECK_OUT, CHECK_IN, PCARD = "7B.CHECK_PAYS_CPTY", "7C.CHECK_CPTY_PAYS", "9B.PCARD_PAYS_CPTY"
CARD_CATS   = ["8B.DEBIT_CARD_PAYS_CPTY", "8C.DEBIT_CARD_CPTY_PAYS"]
ELECTRONIC_OUT_RE = r"^[1-6]B\."   # outbound ACH / wire / RTP: where a check payee could be moved to

ENUM_MAX_DISTINCT = 40
KNOWN_COLS = {"trans_id", "trans_dt", "trans_amt", "category", "src_syst", "payment_rail", "cpty_type",
              "mdm_id_pays", "mdm_id_receives", "pnc_dep_acct_pays", "pnc_dep_acct_receives",
              "customer_name_pays", "customer_name_receives", "unq_cpty_acct_id", "cpty_name", "cpty_fin_entity_name"}

In [ ]:
# ---- NumPy compat shim (must run before any toPandas) ------------------------------------
import numpy as np, warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"pyspark.*")

from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display
import pandas as pd, time, calendar

spark = SparkSession.builder.getOrCreate()
print("Spark", spark.version)
MD = StorageLevel.MEMORY_AND_DISK
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 300);    pd.set_option("display.max_colwidth", 60)

def show(pdf, title, pct=(), money=()):
    print(f"\n■ {title}   [{len(pdf):,} rows]")
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                fmt[c] = "{:.1%}"
        elif c in money:                            fmt[c] = "${:,.0f}"
        elif pd.api.types.is_integer_dtype(pdf[c]): fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):   fmt[c] = "{:,.2f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:    sty = sty.hide(axis="index")
    except AttributeError: sty = sty.hide_index()
    display(sty)

def nz(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

# Masked format signature; delimiters | : / _ - kept.  'CPC|2023-08-30|0001…' → 'A3|D4-D2-D2|D14'
SHAPE_SQL = """CASE WHEN {c} IS NULL THEN 'NULL' ELSE array_join(transform(split({c}, '(?<=[|:/_-])|(?=[|:/_-])'), t ->
  CASE WHEN t rlike '^[|:/_-]$' THEN t WHEN t = '' THEN ''
       WHEN t rlike '^[0-9]+$' THEN concat('D', length(t)) WHEN t rlike '^[A-Za-z]+$' THEN concat('A', length(t))
       WHEN t rlike '^[0-9a-fA-F]+$' THEN concat('H', length(t)) ELSE concat('X', length(t)) END), '') END"""
shape = lambda c: F.expr(SHAPE_SQL.format(c=c))
code_of = lambda cat: cat.split(".")[0]
TIMER = time.time()

## §0 Schema: everything in the table, with columns we have not mapped flagged

In [ ]:
_db, _tbl = TABLE.split(".", 1)
try:    _cat = spark.catalog.listColumns(f"{_db}.{_tbl}")
except Exception: _cat = spark.catalog.listColumns(_tbl, _db)
SCHEMA = pd.DataFrame([{"column": c.name.lower(), "type": c.dataType.lower(), "partition": bool(c.isPartition),
                        "unmapped": c.name.lower() not in KNOWN_COLS} for c in _cat])
show(SCHEMA, f"{TABLE}: {len(SCHEMA)} columns ({SCHEMA.unmapped.sum()} never mapped so far)")
COLS = list(SCHEMA.column); PART = set(SCHEMA[SCHEMA.partition].column)
HAS  = lambda c: c in COLS
for c in (ID_A, ID_B, MERCH_COL):
    print(f"  {c:<18} {'present' if HAS(c) else 'NOT IN TABLE: set the config name'}")

_y, _m = map(int, PROBE_MONTH.split("-"))
LO, HI = f"{PROBE_MONTH}-01", f"{PROBE_MONTH}-{(PROBE_DAYS or calendar.monthrange(_y, _m)[1]):02d}"
raw = spark.table(TABLE).filter(F.col("trans_dt").between(LO, HI))
print("window", LO, "→", HI)

## §1 Population matrix: share of rows where each column is non-null and non-blank

Read it column by column. A column that is 0% everywhere is dead weight; a column populated only in some categories tells you which feed carries it.

In [ ]:
DATA_COLS = [c for c in COLS if c not in PART]
def populated(c):
    return (F.col(c).isNotNull() & (F.trim(F.col(c).cast("string")) != "")).cast("double")

pop = (raw.groupBy("category").agg(F.count("*").alias("rows"), *[F.avg(populated(c)).alias(c) for c in DATA_COLS])
          .orderBy("category").toPandas())
CATS  = list(pop.category)
CODES = [code_of(c) for c in CATS]
ROWS  = dict(zip(pop.category, pop.rows))
show(pd.DataFrame([dict(zip(CODES, pop.rows))]).astype("int64"), "Rows per category (codes)")
M = pop.set_index(pop.category.map(code_of))[DATA_COLS].T.reset_index().rename(columns={"index": "column"})
M.insert(1, "unmapped", M.column.map(lambda c: c not in KNOWN_COLS))
show(M, "Population: column × category", pct=tuple(CODES))

## §2 Enum-like columns (≤ `ENUM_MAX_DISTINCT` values): values per category

The first table lists approximate distinct counts for every string column. That alone shows which columns are ids, which are names, and which are codes.

In [ ]:
STR_COLS = [r.column for r in SCHEMA.itertuples() if r.type.startswith(("string", "varchar", "char")) and not r.partition]
nd = raw.agg(*[F.approx_count_distinct(c).alias(c) for c in STR_COLS]).first().asDict()
ENUMS = [c for c in STR_COLS if 0 < nd[c] <= ENUM_MAX_DISTINCT]
show(pd.DataFrame([{"column": c, "approx_distinct": int(nd[c]), "enum": c in ENUMS, "unmapped": c not in KNOWN_COLS}
                   for c in STR_COLS]).sort_values("approx_distinct"),
     "Distinct values per string column (window)")

ev = (raw.select("category", F.explode(F.array(*[F.struct(F.lit(c).alias("col"), nz(c).alias("val")) for c in ENUMS])).alias("e"))
         .groupBy("category", "e.col", "e.val").count().toPandas())
ev["code"] = ev.category.map(code_of)
ev["share"] = ev["count"] / ev.category.map(ROWS)
for c in ENUMS:
    t = ev[ev.col == c].pivot_table(index="val", columns="code", values="share", aggfunc="sum", fill_value=0.0)
    t = t.reindex(columns=[k for k in CODES if k in t.columns]).reset_index().rename(columns={"val": c})
    t[c] = t[c].fillna("⟨null⟩")
    show(t, f"{c}: share of each category's rows", pct=tuple(k for k in CODES if k in t.columns))

## §3 `unq_cpty_acct_id` (A) vs `unq_cpty_id` (B) vs `merchant_id`

In [ ]:
col_or_null = lambda c: nz(c) if HAS(c) else F.lit(None).cast("string")
S = (raw.select(nz("category").alias("category"), nz("trans_id").alias("trans_id"), F.to_date("trans_dt").alias("dt"),
                F.abs(F.col("trans_amt").cast("double")).alias("amt"),
                col_or_null("mdm_id_pays").alias("mdm_pays"), col_or_null("mdm_id_receives").alias("mdm_recv"),
                col_or_null(ID_A).alias("id_a"), col_or_null(ID_B).alias("id_b"), col_or_null(MERCH_COL).alias("merch"),
                col_or_null("cpty_name").alias("cpty_name"), col_or_null("cpty_type").alias("cpty_type"),
                col_or_null("cpty_fin_entity_name").alias("fi"))
       .withColumn("code", F.split("category", r"\.").getItem(0))
       .withColumn("nm",   F.upper(F.regexp_replace("cpty_name", r"\s+", " ")))
       .withColumn("nm_l", F.regexp_replace(F.upper(F.col("cpty_name")), "[^A-Z0-9]", ""))
       .withColumn("nm_l", F.when(F.col("nm_l") == "", F.lit(None)).otherwise(F.col("nm_l")))
       .persist(MD))
print(f"slim frame: {S.count():,} rows  ({(time.time()-TIMER)/60:,.1f} min so far)")

a, b = F.col("id_a"), F.col("id_b")
cov = (S.groupBy("category").agg(
        F.count("*").alias("rows"),
        F.avg((a.isNotNull() & b.isNotNull()).cast("double")).alias("both"),
        F.avg((a.isNotNull() & b.isNull()).cast("double")).alias("A_only"),
        F.avg((a.isNull() & b.isNotNull()).cast("double")).alias("B_only"),
        F.avg((a.isNull() & b.isNull()).cast("double")).alias("neither"),
        F.avg(F.when(a.isNotNull() & b.isNotNull(), (a == b).cast("double"))).alias("A_eq_B"),
        F.avg(F.when(b.isNotNull(), b.rlike("^[0-9a-fA-F]{32}$").cast("double"))).alias("B_is_hex32"),
        F.avg(F.col("merch").isNotNull().cast("double")).alias("merch_pop"),
        F.approx_count_distinct("id_a").alias("dA"), F.approx_count_distinct("id_b").alias("dB"),
        F.approx_count_distinct("merch").alias("dMerch"), F.approx_count_distinct("nm").alias("dName"))
       .orderBy("category").toPandas())
cov["rows_per_A"] = cov.rows * (cov.both + cov.A_only) / cov.dA.clip(lower=1)
cov["rows_per_B"] = cov.rows * (cov.both + cov.B_only) / cov.dB.clip(lower=1)
cov.insert(0, "code", cov.category.map(code_of))
show(cov.drop(columns="category"), "A vs B coverage per category (d* = approx distinct; rows_per_* = reuse)",
     pct=("both", "A_only", "B_only", "neither", "A_eq_B", "B_is_hex32", "merch_pop"))

In [ ]:
# Formats: top-3 shapes per category for each id-like field, in one pass over the raw rows.
# Includes every unmapped high-cardinality string column, so unfamiliar columns get a look too.
extra = [c for c in STR_COLS if c not in KNOWN_COLS and c not in ENUMS and c not in (ID_B, MERCH_COL)
         and not any(k in c for k in ("name", "desc", "addr"))]
SHAPE_FIELDS = [c for c in ["trans_id", ID_A, ID_B, MERCH_COL] + extra if HAS(c)]
sh = (raw.select("category", F.explode(F.array(*[F.struct(F.lit(c).alias("field"), shape(c).alias("shp")) for c in SHAPE_FIELDS])).alias("s"))
         .groupBy("category", "s.field", "s.shp").count().toPandas())
sh["share"] = sh["count"] / sh.category.map(ROWS)
sh = sh.sort_values(["category", "field", "count"], ascending=[True, True, False])
cell = (sh.groupby(["category", "field"]).head(3)
          .assign(txt=lambda d: d.shp + " " + (d.share * 100).round(0).astype(int).astype(str) + "%")
          .groupby(["category", "field"]).txt.agg(" · ".join).unstack("field"))
cell = cell.reindex(columns=SHAPE_FIELDS).reset_index()
cell.insert(0, "code", cell.category.map(code_of)); cell = cell.drop(columns="category")
show(cell, "Top-3 formats per category (masked shapes, % of category rows)")

In [ ]:
# Is B a hash of something we can see? Test MD5 of candidate inputs on rows where B is 32-hex.
hyp = {"md5(A)": F.md5("id_a"), "md5(merchant_id)": F.md5("merch"), "md5(cpty_name)": F.md5("cpty_name"),
       "md5(norm name)": F.md5("nm"), "md5(alnum name)": F.md5("nm_l"),
       "md5(merchant|name)": F.md5(F.concat_ws("|", "merch", "cpty_name")),
       "md5(merchant+name)": F.md5(F.concat_ws("", "merch", "cpty_name"))}
hx = (S.filter(b.rlike("^[0-9a-fA-F]{32}$"))
        .groupBy("category").agg(F.count("*").alias("hex32_rows"),
            *[F.avg(F.coalesce((e == F.lower(b)).cast("double"), F.lit(0.0))).alias(k) for k, e in hyp.items()])
        .orderBy("category").toPandas())
if len(hx):
    hx.insert(0, "code", hx.category.map(code_of))
    show(hx.drop(columns="category"), "Share of hex-32 B values reproduced by each MD5 hypothesis", pct=tuple(hyp))
else:
    print("no 32-hex values in B: B is not an MD5-style hash")

In [ ]:
# Entity stability: does an id always carry the same name, and does a name always carry the same id?
def stability(idcol):
    g = S.filter(F.col(idcol).isNotNull() & F.col("nm").isNotNull())
    per_id = g.groupBy("category", idcol).agg(F.countDistinct("nm").alias("k"), F.count("*").alias("n"))
    x = per_id.groupBy("category").agg(
          F.count("*").alias("ids"), F.avg((F.col("k") > 1).cast("double")).alias("ids_w_2plus_names"),
          (F.sum(F.when(F.col("k") > 1, F.col("n")).otherwise(0)) / F.sum("n")).alias("rows_on_those_ids"))
    per_nm = g.groupBy("category", "nm").agg(F.countDistinct(idcol).alias("k"))
    y = per_nm.groupBy("category").agg(F.count("*").alias("names"),
          F.avg((F.col("k") > 1).cast("double")).alias("names_w_2plus_ids"), F.avg("k").alias("ids_per_name"))
    return x.join(y, "category").withColumn("id", F.lit(idcol)).toPandas()

st = pd.concat([stability(c) for c in ("id_a", "id_b", "merch")], ignore_index=True)
st.insert(0, "code", st.category.map(code_of))
show(st.drop(columns="category").sort_values(["code", "id"]),
     "Stability: names per id, ids per name (entity-like id: few names per id; persistent id: few ids per name)",
     pct=("ids_w_2plus_names", "rows_on_those_ids", "names_w_2plus_ids"))

# Is the id global? Share of ids seen in 2+ categories, and which category pairs share ids.
rows_sh, pairs = [], []
for idcol in ("id_a", "id_b"):
    x = S.filter(F.col(idcol).isNotNull()).select(F.col(idcol).alias("id"), "code").distinct().persist(MD)
    k = x.groupBy("id").agg(F.count("*").alias("k"))
    r = k.agg(F.count("*").alias("ids"), F.avg((F.col("k") > 1).cast("double")).alias("in_2plus_categories")).first()
    rows_sh.append({"id": idcol, "ids": int(r["ids"] or 0), "in_2plus_categories": r["in_2plus_categories"]})
    p = (x.alias("l").join(x.alias("r"), "id").filter(F.col("l.code") < F.col("r.code"))
           .groupBy(F.col("l.code").alias("cat_1"), F.col("r.code").alias("cat_2")).count()
           .orderBy(F.desc("count")).limit(12).toPandas())
    p.insert(0, "id", idcol); pairs.append(p); x.unpersist()
show(pd.DataFrame(rows_sh), "Ids shared across categories", pct=("in_2plus_categories",))
show(pd.concat(pairs, ignore_index=True).rename(columns={"count": "shared_ids"}), "Top category pairs sharing the same id value")

## §4 Checks

**7B (outbound).** Two questions: is the `trans_id` tail an account or an item sequence, and how usable are payee names as ids?
**Conversion sizing:** check payees whose (letters+digits) name is also paid through outbound ACH, wire or RTP, either by **any** PNC customer or by **the same** customer.
**7C (inbound).** Which banks the checks are drawn on, and the on-us share, i.e. checks drawn on PNC and recorded as counterparty.

In [ ]:
c7 = S.filter(F.col("category") == CHECK_OUT)
parts = F.split("trans_id", r"\|")
t7 = c7.select(parts.getItem(0).alias("p0"), F.size(parts).alias("nparts"), F.element_at(parts, -1).alias("tail"),
               "dt", "nm", "mdm_pays", "amt").persist(MD)

g = t7.groupBy("tail").agg(F.count("*").alias("n"), F.countDistinct("nm").alias("names"),
                           F.countDistinct("mdm_pays").alias("payers"), F.countDistinct("dt").alias("dates"))
r = g.agg(F.count("*").alias("distinct_tails"), F.sum("n").alias("rows"),
          F.sum((F.col("n") > 1).cast("long")).alias("tails_repeating"),
          F.avg(F.when(F.col("n") > 1, (F.col("names") == 1).cast("double"))).alias("repeat_same_payee"),
          F.avg(F.when(F.col("n") > 1, (F.col("payers") == 1).cast("double"))).alias("repeat_same_payer"),
          F.avg(F.when(F.col("n") > 1, (F.col("dates") > 1).cast("double"))).alias("repeat_across_dates")).toPandas()
r["tails_per_row"] = r.distinct_tails / r.rows
show(r, f"{CHECK_OUT}: trans_id tail. ≈1 tail per row → item sequence; repeats with same payee across dates → account-like",
     pct=("repeat_same_payee", "repeat_same_payer", "repeat_across_dates", "tails_per_row"))

rng = (t7.filter(F.col("tail").rlike("^[0-9]+$")).groupBy("dt")
         .agg(F.count("*").alias("checks"), F.min(F.col("tail").cast("decimal(20,0)")).alias("min_tail"),
              F.max(F.col("tail").cast("decimal(20,0)")).alias("max_tail"))
         .withColumn("span", F.col("max_tail") - F.col("min_tail")).orderBy("dt").limit(10).toPandas())
rng["span_per_check"] = rng.span.astype(float) / rng.checks
show(rng.astype({"min_tail": "int64", "max_tail": "int64", "span": "int64"}),
     "Tail range per posting date (a tight range that climbs day to day = processing sequence)")
show(t7.groupBy("p0", "nparts").count().orderBy(F.desc("count")).limit(5).toPandas(), "trans_id prefix / segment count")

In [ ]:
BIZ = (r"\b(INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LP|LLP|PC|PLLC|PA|NA|BANK|TRUST|FUND|FOUNDATION|"
       r"ASSOC|ASSOCIATION|SERVICES?|GROUP|HOLDINGS|PARTNERS|UNIVERSITY|COLLEGE|SCHOOL|DISTRICT|CITY|COUNTY|STATE|"
       r"TOWNSHIP|BOROUGH|HOSPITAL|CENTER|CHURCH|DEPT|DEPARTMENT|TREASURER|INSURANCE|SUPPLY|ENTERPRISES|INTERNATIONAL|AUTHORITY)\b")
c7b = c7.withColumn("biz", F.col("nm").rlike(BIZ))
pp = c7b.filter(F.col("nm").isNotNull()).groupBy("mdm_pays").agg(F.countDistinct("nm").alias("payees"), F.count("*").alias("checks"))
q = pp.agg(F.count("*").alias("payers"),
           F.expr("percentile_approx(payees, array(0.5, 0.9, 0.99))").alias("payees_p50_p90_p99"),
           F.expr("percentile_approx(checks, array(0.5, 0.9, 0.99))").alias("checks_p50_p90_p99")).toPandas()
n7 = c7b.agg(F.count("*").alias("rows"), F.avg(F.col("nm").isNotNull().cast("double")).alias("name_populated"),
             F.countDistinct("nm").alias("distinct_names"), F.countDistinct("nm_l").alias("distinct_alnum_names"),
             F.avg(F.col("biz").cast("double")).alias("business_like_rows"),
             (F.sum(F.when(F.col("biz"), F.col("amt")).otherwise(0)) / F.sum("amt")).alias("business_like_usd")).toPandas()
show(n7, f"{CHECK_OUT}: payee names (business-like = legal/entity token in name; heuristic)",
     pct=("name_populated", "business_like_rows", "business_like_usd"))
q["payees_p50_p90_p99"] = q.payees_p50_p90_p99.map(lambda v: " / ".join(str(int(x)) for x in v))
q["checks_p50_p90_p99"] = q.checks_p50_p90_p99.map(lambda v: " / ".join(str(int(x)) for x in v))
show(q, "Per payer: distinct payees and checks in the window")

elec = S.filter(F.col("category").rlike(ELECTRONIC_OUT_RE) & F.col("nm_l").isNotNull())
e_any  = elec.select("nm_l").distinct().withColumn("f_any", F.lit(1))
e_same = elec.select("mdm_pays", "nm_l").distinct().withColumn("f_same", F.lit(1))
m = (c7b.filter(F.col("nm_l").isNotNull()).join(e_any, "nm_l", "left").join(e_same, ["mdm_pays", "nm_l"], "left")
        .agg(F.count("*").alias("rows"), F.sum("amt").alias("usd"),
             F.avg(F.col("f_any").isNotNull().cast("double")).alias("payee_paid_electronically_by_anyone"),
             (F.sum(F.when(F.col("f_any").isNotNull(), F.col("amt")).otherwise(0)) / F.sum("amt")).alias("…_usd"),
             F.avg(F.col("f_same").isNotNull().cast("double")).alias("same_customer_also_pays_electronically"),
             (F.sum(F.when(F.col("f_same").isNotNull(), F.col("amt")).otherwise(0)) / F.sum("amt")).alias("…same_usd"),
             F.avg(F.when(F.col("biz"), F.col("f_any").isNotNull().cast("double"))).alias("business_payees_paid_electronically"))
        .toPandas())
show(m, "Check → electronic conversion sizing (exact alnum-name match; undercounts spelling variants)",
     pct=tuple(c for c in m.columns if c not in ("rows", "usd")), money=("usd",))

In [ ]:
c7c = (S.filter(F.col("category") == CHECK_IN)
         .withColumn("acct", F.coalesce("id_a", "id_b"))
         .withColumn("aba", F.split("acct", "-").getItem(0)))
tot = c7c.agg(F.count("*").alias("rows"), F.sum("amt").alias("usd"),
              F.avg(F.col("nm").isNotNull().cast("double")).alias("name_populated"),
              F.avg(F.col("fi").isNotNull().cast("double")).alias("fi_populated"),
              F.avg(F.col("aba").rlike("^[0-9]{9}$").cast("double")).alias("aba_like_prefix"),
              F.avg(F.coalesce(F.col("fi").rlike("(?i)\\bPNC\\b").cast("double"), F.lit(0.0))).alias("on_us_rows"),
              (F.sum(F.when(F.col("fi").rlike("(?i)\\bPNC\\b"), F.col("amt")).otherwise(0)) / F.sum("amt")).alias("on_us_usd")).toPandas()
show(tot, f"{CHECK_IN}: drawee bank coverage and on-us share (FI name contains PNC)",
     pct=("name_populated", "fi_populated", "aba_like_prefix", "on_us_rows", "on_us_usd"), money=("usd",))
top = (c7c.groupBy("aba", "fi").agg(F.count("*").alias("rows"), F.sum("amt").alias("usd"))
          .orderBy(F.desc("rows")).limit(12).toPandas())
top["share_rows"] = top.rows / tot.rows[0]
show(top, "Top drawee routing numbers (routing numbers are public identifiers)", pct=("share_rows",), money=("usd",))

## §5 PCARD and debit card: `merchant_id` vs `unq_cpty_id` vs name

In [ ]:
def fan(df, x, y):
    k = df.filter(F.col(x).isNotNull() & F.col(y).isNotNull()).groupBy(x).agg(F.countDistinct(y).alias("k"))
    return k.agg(F.avg((F.col("k") > 1).cast("double"))).first()[0]

cards = []
for cat in [PCARD] + CARD_CATS:
    d = S.filter(F.col("category") == cat).persist(MD)
    if d.limit(1).count() == 0:
        d.unpersist(); continue
    cards.append({"category": cat,
        "merch→2+ B":      fan(d, "merch", "id_b"), "B→2+ merch":   fan(d, "id_b", "merch"),
        "merch→2+ names":  fan(d, "merch", "nm"),   "B→2+ names":   fan(d, "id_b", "nm"),
        "name→2+ merch":   fan(d, "nm", "merch"),   "name→2+ B":    fan(d, "nm", "id_b"),
        "cpty_type values": ", ".join(r[0] or "⟨null⟩" for r in d.select("cpty_type").distinct().limit(6).collect())})
    d.unpersist()
cp = pd.DataFrame(cards)
show(cp, "Card ids: share of values mapping to 2+ of the other (0% = one-to-one)",
     pct=tuple(c for c in cp.columns if "→" in c))

## §6 `*_orig` columns vs their base column

In [ ]:
ORIG = [(c[:-5], c) for c in COLS if c.endswith("_orig") and c[:-5] in COLS]
if ORIG:
    aggs = []
    for base, orig in ORIG:
        bb, oo = nz(base), nz(orig)
        aggs += [F.avg((bb.isNotNull() & oo.isNotNull()).cast("double")).alias(f"{orig}|both"),
                 F.avg(F.when(bb.isNotNull() & oo.isNotNull(), (bb != oo).cast("double"))).alias(f"{orig}|differ"),
                 F.avg((bb.isNull() & oo.isNotNull()).cast("double")).alias(f"{orig}|orig_only"),
                 F.avg((bb.isNotNull() & oo.isNull()).cast("double")).alias(f"{orig}|base_only")]
    o = raw.groupBy("category").agg(*aggs).orderBy("category").toPandas()
    o.insert(0, "code", o.category.map(code_of))
    show(o.drop(columns="category"), f"*_orig vs base: {', '.join(x for _, x in ORIG)}",
         pct=tuple(c for c in o.columns if "|" in c))
else:
    print("no *_orig columns with a matching base column")

for _df in ["t7", "S"]:
    if _df in globals(): globals()[_df].unpersist()
print(f"total runtime {(time.time()-TIMER)/60:,.1f} min")

### What to screenshot
§0 schema · §1 population matrix · §2 distinct-count table plus the `cpty_type` table · §3 coverage, formats, MD5 test, stability, sharing ·
§4 all 7B tables plus the 7C tables · §5 card table · §6.